In [ ]:
from datetime import date
import polars as pl
from data_engineering.adsb.read_adsb import read_adsb
df_adsb = read_adsb(date(2026,3,1))

## Callsign noise

Treat two effects separately:

- **availability noise**: `callsign` changes between null and non-null; measured by `missing_rate` and `presence_flips`.
- **identity noise**: the non-null value changes. Consecutive identical values are compressed into runs, and `A → B → A` is counted as a flip-flop.

Each ICAO is split into tracks after a 30-minute observation gap so callsigns from separate flights are less likely to be compared. Nulls are ignored when measuring identity changes.

In [ ]:
TRACK_GAP_MINUTES = 30
SHORT_FLIP_SECONDS = 60
track_keys = ["icao", "track_id"]

df_events = (
    df_adsb
    .with_columns(
        pl.when(
            pl.col("callsign").is_null()
            | pl.col("callsign").str.strip_chars().eq("")
        )
        .then(None)
        .otherwise(pl.col("callsign").str.strip_chars().str.to_uppercase())
        .alias("callsign_clean")
    )
    .sort(["icao", "time"])
    .with_columns(
        pl.col("time")
        .diff()
        .over("icao")
        .gt(pl.duration(minutes=TRACK_GAP_MINUTES))
        .fill_null(True)
        .cast(pl.UInt32)
        .cum_sum()
        .over("icao")
        .alias("track_id")
    )
    .with_columns(
        (
            pl.col("callsign_clean").is_not_null()
            != pl.col("callsign_clean").is_not_null().shift(1).over(track_keys)
        )
        .fill_null(False)
        .alias("presence_flip")
    )
)

In [ ]:
# Compress non-null observations into callsign runs within each continuous track.
df_known = (
    df_events
    .filter(pl.col("callsign_clean").is_not_null())
    .with_columns(
        (
            pl.col("callsign_clean")
            != pl.col("callsign_clean").shift(1).over(track_keys)
        )
        .fill_null(True)
        .alias("new_run")
    )
    .with_columns(
        pl.col("new_run")
        .cast(pl.UInt32)
        .cum_sum()
        .over(track_keys)
        .alias("run_id")
    )
)

callsign_runs = (
    df_known
    .group_by(track_keys + ["run_id"], maintain_order=True)
    .agg(
        pl.col("callsign_clean").first().alias("callsign"),
        pl.col("time").min().alias("run_start"),
        pl.col("time").max().alias("run_end"),
        pl.len().alias("run_samples"),
    )
    .sort(track_keys + ["run_start"])
    .with_columns(
        pl.col("callsign").shift(2).over(track_keys).alias("callsign_before"),
        pl.col("callsign").shift(1).over(track_keys).alias("middle_callsign"),
        pl.col("run_start").shift(1).over(track_keys).alias("middle_start"),
        pl.col("run_end").shift(1).over(track_keys).alias("middle_end"),
        pl.col("run_samples").shift(1).over(track_keys).alias("middle_samples"),
    )
    .with_columns(
        (
            (pl.col("callsign") == pl.col("callsign_before"))
            & (pl.col("callsign") != pl.col("middle_callsign"))
        )
        .fill_null(False)
        .alias("is_flip_flop"),
        (pl.col("middle_end") - pl.col("middle_start"))
        .dt.total_seconds()
        .alias("middle_duration_s"),
    )
)

In [ ]:
track_stats = (
    df_events
    .group_by(track_keys)
    .agg(
        pl.col("time").min().alias("start"),
        pl.col("time").max().alias("end"),
        pl.len().alias("rows"),
        pl.col("callsign_clean").is_not_null().sum().alias("known_rows"),
        pl.col("callsign_clean").is_null().mean().alias("missing_rate"),
        pl.col("callsign_clean").drop_nulls().n_unique().alias("unique_callsigns"),
        pl.col("presence_flip").sum().alias("presence_flips"),
    )
)

run_stats = (
    callsign_runs
    .group_by(track_keys)
    .agg(
        (pl.len() - 1).alias("callsign_changes"),
        pl.col("is_flip_flop").sum().alias("flip_flops"),
        (
            pl.col("is_flip_flop")
            & (
                (pl.col("middle_samples") == 1)
                | (pl.col("middle_duration_s") <= SHORT_FLIP_SECONDS)
            )
        )
        .sum()
        .alias("short_flip_flops"),
    )
)

track_noise = (
    track_stats
    .join(run_stats, on=track_keys, how="left")
    .with_columns(
        pl.col(["callsign_changes", "flip_flops", "short_flip_flops"])
        .fill_null(0),
        pl.when(pl.col("known_rows") > 1)
        .then(pl.col("callsign_changes") / (pl.col("known_rows") - 1) * 100)
        .otherwise(0.0)
        .alias("changes_per_100_known_rows"),
    )
    .sort(
        ["short_flip_flops", "flip_flops", "callsign_changes", "presence_flips"],
        descending=True,
    )
)

track_noise.head(50)

icao,track_id,start,end,rows,known_rows,missing_rate,unique_callsigns,presence_flips,callsign_changes,flip_flops,short_flip_flops,changes_per_100_known_rows
str,u32,datetime[ms],datetime[ms],u32,u32,f64,u32,u32,u32,u32,u32,f64
"""7cbbbd""",1,2026-03-01 00:00:11.380,2026-03-01 00:14:46.470,375,93,0.752,2,186,35,34,33,38.043478
"""49f0c1""",1,2026-03-01 06:37:57.140,2026-03-01 08:43:03.810,864,216,0.75,2,431,43,42,30,20.0
"""7cf7ca""",1,2026-03-01 00:00:00.240,2026-03-01 15:18:13.720,8688,2172,0.75,2,4343,50,49,26,2.303086
"""a74ecc""",3,2026-03-01 18:20:55.600,2026-03-01 23:59:51.750,1127,269,0.761313,2,538,32,31,15,11.940299
"""000001""",4,2026-03-01 11:21:00.200,2026-03-01 12:36:56.450,813,203,0.750308,3,405,20,17,12,9.90099
…,…,…,…,…,…,…,…,…,…,…,…,…
"""3c65ce""",5,2026-03-01 19:38:21.250,2026-03-01 20:27:41.410,601,149,0.75208,2,298,8,7,6,5.405405
"""71c592""",6,2026-03-01 07:34:12.350,2026-03-01 08:25:31.960,537,132,0.75419,2,264,8,7,6,6.10687
"""7892a4""",2,2026-03-01 07:47:07.940,2026-03-01 09:37:17.500,578,127,0.780277,2,254,8,7,6,6.349206


In [ ]:
# One-row overview for the day.
overview = df_events.select(
    pl.len().alias("rows"),
    pl.col("callsign_clean").is_null().mean().alias("missing_rate"),
    pl.col("presence_flip").sum().alias("null_nonnull_flips"),
).hstack(
    track_noise.select(
        pl.len().alias("tracks"),
        (pl.col("unique_callsigns") > 1).sum().alias("tracks_with_multiple_callsigns"),
        pl.col("callsign_changes").sum().alias("callsign_changes"),
        pl.col("flip_flops").sum().alias("flip_flops"),
        pl.col("short_flip_flops").sum().alias("short_flip_flops"),
    )
)
overview

rows,missing_rate,null_nonnull_flips,tracks,tracks_with_multiple_callsigns,callsign_changes,flip_flops,short_flip_flops
u32,f64,u32,u32,u32,u32,u32,u32
85799483,0.761459,40873229,181828,4872,9402,4149,3464


In [ ]:
# Exact A -> B -> A events. The middle value is the likely noisy value when it
# appeared once or lasted no more than SHORT_FLIP_SECONDS.
flip_flop_details = (
    callsign_runs
    .filter(pl.col("is_flip_flop"))
    .select(
        "icao",
        "track_id",
        pl.col("callsign_before").alias("A"),
        pl.col("middle_callsign").alias("B"),
        "middle_start",
        "middle_end",
        "middle_samples",
        "middle_duration_s",
        pl.col("run_start").alias("A_returned_at"),
        (
            (pl.col("middle_samples") == 1)
            | (pl.col("middle_duration_s") <= SHORT_FLIP_SECONDS)
        ).alias("is_short"),
    )
    .sort(["is_short", "middle_duration_s"], descending=[True, False])
)

flip_flop_details.head(100)

icao,track_id,A,B,middle_start,middle_end,middle_samples,middle_duration_s,A_returned_at,is_short
str,u32,str,str,datetime[ms],datetime[ms],u32,i64,datetime[ms],bool
"""4bcdb6""",1,"""SXS3A""","""@@@@@@@@""",2026-03-01 11:28:53.460,2026-03-01 11:28:53.460,1,0,2026-03-01 11:29:04.300,true
"""484fde""",2,"""KLM40D""","""@@@@@@@@""",2026-03-01 13:09:42.890,2026-03-01 13:09:42.890,1,0,2026-03-01 13:09:49.470,true
"""000001""",4,"""SAR05""","""WABEL02""",2026-03-01 11:39:03.290,2026-03-01 11:39:03.290,1,0,2026-03-01 11:39:11.640,true
"""e491a1""",7,"""PTXPC4""","""PTXPC""",2026-03-01 20:32:36.510,2026-03-01 20:32:36.510,1,0,2026-03-01 20:33:48.380,true
"""000001""",4,"""SAR05""","""WABEL02""",2026-03-01 11:39:40.260,2026-03-01 11:39:40.260,1,0,2026-03-01 11:39:52.220,true
…,…,…,…,…,…,…,…,…,…
"""344648""",5,"""IBE0485""","""VOE7ZE""",2026-03-01 21:21:00.770,2026-03-01 21:21:00.770,1,0,2026-03-01 21:21:47.580,true
"""344695""",2,"""VLG9ST""","""@@@@@@@@""",2026-03-01 11:21:26.230,2026-03-01 11:21:26.230,1,0,2026-03-01 11:22:04.540,true
"""c07ef2""",1,"""WJA247""","""@@@@@@@@""",2026-03-01 03:08:08.100,2026-03-01 03:08:08.100,1,0,2026-03-01 03:08:31.590,true
